# ETL Datamart de Ventas (SQL Server en GCP)\n\nEste notebook espera a que la base `ventas_dm` y la tabla `dwh.fact_ventas` esten listas, luego ejecuta una carga incremental desde `data/ventas_raw.csv`.

In [ ]:
import time\nfrom pathlib import Path\nimport urllib.parse\n\nimport pandas as pd\nfrom sqlalchemy import create_engine, text\n\nSQL_SERVER_HOST = "REEMPLAZAR_CON_IP_PUBLICA"\nSQL_SERVER_PORT = 1433\nSQL_SERVER_USER = "sa"\nSQL_SERVER_PASSWORD = "REEMPLAZAR_PASSWORD"\nSQL_SERVER_DRIVER = "ODBC Driver 18 for SQL Server"\nTARGET_DB = "ventas_dm"\n

In [ ]:
def make_engine(database: str):\n    conn = (\n        f"DRIVER={{{SQL_SERVER_DRIVER}}};"\n        f"SERVER={SQL_SERVER_HOST},{SQL_SERVER_PORT};"\n        f"DATABASE={database};"\n        f"UID={SQL_SERVER_USER};PWD={SQL_SERVER_PASSWORD};"\n        "Encrypt=yes;TrustServerCertificate=yes;"\n    )\n    return create_engine(\n        f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(conn)}",\n        future=True,\n        fast_executemany=True,\n    )\n\n\ndef wait_for_datamart(timeout_sec: int = 1200, sleep_sec: int = 10):\n    engine_master = make_engine("master")\n    started = time.time()\n\n    while time.time() - started < timeout_sec:\n        try:\n            with engine_master.connect() as conn:\n                db_exists = conn.execute(text("SELECT DB_ID(:db)"), {"db": TARGET_DB}).scalar()\n                if db_exists is None:\n                    time.sleep(sleep_sec)\n                    continue\n\n            engine_target = make_engine(TARGET_DB)\n            with engine_target.connect() as conn:\n                tbl_exists = conn.execute(text("SELECT OBJECT_ID('dwh.fact_ventas','U')")).scalar()\n                if tbl_exists is not None:\n                    print("Datamart listo para carga.")\n                    return engine_target\n        except Exception as ex:\n            print(f"Esperando SQL Server... {ex}")\n\n        time.sleep(sleep_sec)\n\n    raise TimeoutError("Tiempo de espera agotado para SQL Server/datamart.")\n\n\nengine = wait_for_datamart()\n

In [ ]:
project_root = Path.cwd()\nif not (project_root / "data").exists() and (project_root.parent / "data").exists():\n    project_root = project_root.parent\n\ncsv_path = project_root / "data" / "ventas_raw.csv"\ndf = pd.read_csv(csv_path)\n\ndf["fecha_venta"] = pd.to_datetime(df["fecha_venta"], errors="coerce").dt.date\nfor c in ["cantidad", "linea_factura"]:\n    df[c] = pd.to_numeric(df[c], errors="coerce")\nfor c in ["precio_unitario", "descuento"]:\n    df[c] = pd.to_numeric(df[c], errors="coerce")\n\ntext_cols = [\n    "codigo_producto", "nombre_producto", "categoria_producto",\n    "codigo_cliente", "nombre_cliente", "segmento_cliente",\n    "codigo_tienda", "nombre_tienda", "region_tienda", "numero_factura"\n]\nfor c in text_cols:\n    df[c] = df[c].astype(str).str.strip()\n\ndf = df.dropna(subset=["fecha_venta", "cantidad", "precio_unitario", "linea_factura"])\ndf = df[(df["cantidad"] > 0) & (df["precio_unitario"] >= 0) & (df["descuento"] >= 0)]\ndf = df.drop_duplicates()\ndf.shape\n

In [ ]:
with engine.begin() as conn:\n    conn.exec_driver_sql("TRUNCATE TABLE dwh.stg_ventas;")\n\ndf.to_sql(\n    name="stg_ventas",\n    schema="dwh",\n    con=engine,\n    if_exists="append",\n    index=False,\n    chunksize=1000,\n    method="multi",\n)\n\nwith engine.begin() as conn:\n    conn.exec_driver_sql('''SET NOCOUNT ON;

TRUNCATE TABLE dwh.stg_ventas;

INSERT INTO dwh.dim_fecha (key_fecha, fecha, anio, trimestre, mes, nombre_mes, dia, nombre_dia, es_fin_semana)
SELECT DISTINCT
  CAST(CONVERT(VARCHAR(8), s.fecha_venta, 112) AS INT) AS key_fecha,
  s.fecha_venta,
  DATEPART(YEAR, s.fecha_venta) AS anio,
  DATEPART(QUARTER, s.fecha_venta) AS trimestre,
  DATEPART(MONTH, s.fecha_venta) AS mes,
  DATENAME(MONTH, s.fecha_venta) AS nombre_mes,
  DATEPART(DAY, s.fecha_venta) AS dia,
  DATENAME(WEEKDAY, s.fecha_venta) AS nombre_dia,
  CASE WHEN DATEPART(WEEKDAY, s.fecha_venta) IN (1,7) THEN 1 ELSE 0 END AS es_fin_semana
FROM dwh.stg_ventas s
LEFT JOIN dwh.dim_fecha d ON d.key_fecha = CAST(CONVERT(VARCHAR(8), s.fecha_venta, 112) AS INT)
WHERE d.key_fecha IS NULL;

MERGE dwh.dim_producto AS t
USING (
  SELECT DISTINCT codigo_producto, nombre_producto, categoria_producto
  FROM dwh.stg_ventas
) AS s
ON t.codigo_producto = s.codigo_producto
WHEN MATCHED AND (t.nombre_producto <> s.nombre_producto OR t.categoria_producto <> s.categoria_producto) THEN
  UPDATE SET t.nombre_producto = s.nombre_producto, t.categoria_producto = s.categoria_producto
WHEN NOT MATCHED THEN
  INSERT (codigo_producto, nombre_producto, categoria_producto)
  VALUES (s.codigo_producto, s.nombre_producto, s.categoria_producto);

MERGE dwh.dim_cliente AS t
USING (
  SELECT DISTINCT codigo_cliente, nombre_cliente, segmento_cliente
  FROM dwh.stg_ventas
) AS s
ON t.codigo_cliente = s.codigo_cliente
WHEN MATCHED AND (t.nombre_cliente <> s.nombre_cliente OR t.segmento_cliente <> s.segmento_cliente) THEN
  UPDATE SET t.nombre_cliente = s.nombre_cliente, t.segmento_cliente = s.segmento_cliente
WHEN NOT MATCHED THEN
  INSERT (codigo_cliente, nombre_cliente, segmento_cliente)
  VALUES (s.codigo_cliente, s.nombre_cliente, s.segmento_cliente);

MERGE dwh.dim_tienda AS t
USING (
  SELECT DISTINCT codigo_tienda, nombre_tienda, region_tienda
  FROM dwh.stg_ventas
) AS s
ON t.codigo_tienda = s.codigo_tienda
WHEN MATCHED AND (t.nombre_tienda <> s.nombre_tienda OR t.region_tienda <> s.region_tienda) THEN
  UPDATE SET t.nombre_tienda = s.nombre_tienda, t.region_tienda = s.region_tienda
WHEN NOT MATCHED THEN
  INSERT (codigo_tienda, nombre_tienda, region_tienda)
  VALUES (s.codigo_tienda, s.nombre_tienda, s.region_tienda);

WITH src AS (
  SELECT
    CAST(CONVERT(VARCHAR(8), s.fecha_venta, 112) AS INT) AS key_fecha,
    p.key_producto,
    c.key_cliente,
    t.key_tienda,
    s.numero_factura,
    s.linea_factura,
    CAST(s.cantidad AS INT) AS cantidad,
    CAST(s.precio_unitario AS DECIMAL(18,2)) AS precio_unitario,
    CAST(s.descuento AS DECIMAL(18,2)) AS descuento
  FROM dwh.stg_ventas s
  INNER JOIN dwh.dim_producto p ON p.codigo_producto = s.codigo_producto
  INNER JOIN dwh.dim_cliente c ON c.codigo_cliente = s.codigo_cliente
  INNER JOIN dwh.dim_tienda t ON t.codigo_tienda = s.codigo_tienda
)
MERGE dwh.fact_ventas AS tgt
USING src
  ON tgt.key_fecha = src.key_fecha
 AND tgt.key_producto = src.key_producto
 AND tgt.key_cliente = src.key_cliente
 AND tgt.key_tienda = src.key_tienda
 AND tgt.numero_factura = src.numero_factura
 AND tgt.linea_factura = src.linea_factura
WHEN MATCHED THEN
  UPDATE SET
    tgt.cantidad = src.cantidad,
    tgt.precio_unitario = src.precio_unitario,
    tgt.descuento = src.descuento,
    tgt.fecha_carga = SYSUTCDATETIME()
WHEN NOT MATCHED THEN
  INSERT (key_fecha, key_producto, key_cliente, key_tienda, numero_factura, linea_factura, cantidad, precio_unitario, descuento)
  VALUES (src.key_fecha, src.key_producto, src.key_cliente, src.key_tienda, src.numero_factura, src.linea_factura, src.cantidad, src.precio_unitario, src.descuento);
''')\n\nprint("ETL completado.")\n

In [ ]:
query = '''\nSELECT TOP (20)\n    f.key_fact_venta,\n    d.fecha,\n    p.nombre_producto,\n    c.nombre_cliente,\n    t.nombre_tienda,\n    f.cantidad,\n    f.precio_unitario,\n    f.descuento,\n    f.monto_neto\nFROM dwh.fact_ventas f\nJOIN dwh.dim_fecha d ON d.key_fecha = f.key_fecha\nJOIN dwh.dim_producto p ON p.key_producto = f.key_producto\nJOIN dwh.dim_cliente c ON c.key_cliente = f.key_cliente\nJOIN dwh.dim_tienda t ON t.key_tienda = f.key_tienda\nORDER BY f.key_fact_venta DESC;\n'''\npd.read_sql(query, engine)\n